In [5]:
!pip install -q "numpy<2.0.0" terratorch

In [1]:
import rasterio, numpy as np
from rasterio.warp import reproject, Resampling
from pathlib import Path

BASE_DIR       = Path('/content/drive/MyDrive/prithvi_fault')
HLS_DIR        = BASE_DIR / 'data' / 'raw' / 'hls'
PROC_DIR       = BASE_DIR / 'data' / 'processed'
FAULT_MASK_TIF = PROC_DIR / 'fault_mask_parkfield.tif'  # buffer=10m 마스크

# 256×256 패치 디렉토리
PATCH_256_DIR  = BASE_DIR / 'data' / 'patches' / 'parkfield_hls_256'
PATCH_256_DIR.mkdir(parents=True, exist_ok=True)

# ── 마스크 로드 ───────────────────────────────────────────────
with rasterio.open(FAULT_MASK_TIF) as src:
    MASK_CRS   = src.crs
    MASK_TRANS = src.transform
    MASK_W     = src.width
    MASK_H     = src.height
    fault_mask = src.read(1).astype(np.uint8)
print(f'마스크: {MASK_W}×{MASK_H}, fault={fault_mask.mean()*100:.3f}%')

# ── HLS 스택 로드 ─────────────────────────────────────────────
TARGET_BANDS = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
season_map   = {
    'spring': '2022091',
    'summer': '2022186',
    'fall':   '2022246',
    'winter': '2022336',
}

season_arrays = []
for sn, date_str in season_map.items():
    bands = []
    for band in TARGET_BANDS:
        matches = list((HLS_DIR / sn).glob(f'*{date_str}*.{band}.*'))
        if not matches:
            bands.append(np.zeros((MASK_H, MASK_W), dtype=np.float32))
            continue
        with rasterio.open(matches[0]) as src:
            dst = np.zeros((MASK_H, MASK_W), dtype=np.float32)
            reproject(
                source=src.read(1).astype(np.float32),
                destination=dst,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=MASK_TRANS, dst_crs=MASK_CRS,
                resampling=Resampling.bilinear
            )
            valid = dst[dst > 0]
            if len(valid) > 0 and valid.max() <= 1.0:
                dst = dst * 10000.0
        bands.append(dst)
    season_arrays.append(np.stack(bands, axis=0))
    print(f'{sn}: 완료')

hls_stack = np.stack(season_arrays, axis=0)  # (4, 6, H, W)
print(f'HLS stack: {hls_stack.shape}')

# ── 256×256 패치 추출 ─────────────────────────────────────────
PATCH_SIZE = 256  # ✅ 128 → 256
H, W = MASK_H, MASK_W
all_imgs, all_masks = [], []

for r in range(0, H - PATCH_SIZE + 1, PATCH_SIZE):
    for c in range(0, W - PATCH_SIZE + 1, PATCH_SIZE):
        img_p  = hls_stack[:, :, r:r+PATCH_SIZE, c:c+PATCH_SIZE]
        mask_p = fault_mask[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
        if img_p.shape != (4, 6, PATCH_SIZE, PATCH_SIZE):
            continue
        if np.isnan(img_p).mean() > 0.05:
            continue
        all_imgs.append(np.nan_to_num(img_p).astype(np.float32))
        all_masks.append(mask_p.astype(np.uint8))

imgs  = np.stack(all_imgs)
masks = np.stack(all_masks)
has_fault = masks.sum(axis=(1,2)) > 0
print(f'\n총 패치: {len(imgs)}')
print(f'fault 패치: {has_fault.sum()} ({has_fault.mean()*100:.1f}%)')
fp_px = masks[has_fault].sum(axis=(1,2)) if has_fault.sum() > 0 else np.array([0])
print(f'패치당 fault 픽셀: {fp_px.mean():.1f} ({fp_px.mean()/(256*256)*100:.2f}%)')

# ── Split + 저장 ──────────────────────────────────────────────
fi  = np.where(has_fault)[0]
bi  = np.where(~has_fault)[0]
rng = np.random.default_rng(42)
rng.shuffle(fi); rng.shuffle(bi)

def spl(idx):
    n=len(idx); a=int(n*0.70); b=int(n*0.15)
    return idx[:a], idx[a:a+b], idx[a+b:]

f_tr,f_va,f_te = spl(fi)
b_tr,b_va,b_te = spl(bi)

for name, idx in [('train', np.concatenate([f_tr,b_tr])),
                  ('val',   np.concatenate([f_va,b_va])),
                  ('test',  np.concatenate([f_te,b_te]))]:
    if len(idx) == 0:
        print(f'  {name}: 0개')
        continue
    np.savez_compressed(PATCH_256_DIR/f'{name}.npz',
                        images=imgs[idx], masks=masks[idx])
    nf = int((masks[idx].sum(axis=(1,2))>0).sum())
    print(f'  {name}: {len(idx)}개 (fault={nf}, {nf/len(idx)*100:.0f}%)')

print(f'\n✅ 완료: {PATCH_256_DIR}')

마스크: 8098×7980, fault=1.220%
spring: 완료
summer: 완료
fall: 완료
winter: 완료
HLS stack: (4, 6, 7980, 8098)

총 패치: 961
fault 패치: 102 (10.6%)
패치당 fault 픽셀: 7598.8 (11.59%)
  train: 672개 (fault=71, 11%)
  val: 143개 (fault=15, 10%)
  test: 146개 (fault=16, 11%)

✅ 완료: /content/drive/MyDrive/prithvi_fault/data/patches/parkfield_hls_256


In [2]:
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

HLS_MEAN = np.array([1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0], dtype=np.float32)
HLS_STD  = np.array([2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0], dtype=np.float32)

PROJECT_DIR = Path('/content/drive/MyDrive/prithvi_fault')
PATCH_DIR   = PROJECT_DIR / 'data' / 'patches' / 'parkfield_hls_256'
CKPT_DIR    = PROJECT_DIR / 'checkpoints_prithvi_256'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

class HLSDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        data  = np.load(npz_path)
        imgs  = data['images'].astype(np.float32)   # (N, 4, 6, H, W)
        masks = data['masks'].astype(np.int64)       # (N, H, W)
        imgs  = np.nan_to_num(imgs, nan=0.0)
        mean  = HLS_MEAN[None, None, :, None, None]
        std   = HLS_STD[None, None, :, None, None]
        imgs  = (imgs - mean) / (std + 1e-8)

        self.images    = torch.from_numpy(imgs).float()
        self.masks     = torch.from_numpy(masks)
        self.augment   = augment
        self.has_fault = (masks.sum(axis=(1,2)) > 0)
        n, nf = len(self.images), int(self.has_fault.sum())
        print(f'  {Path(npz_path).name}: {n}개, fault={nf}({nf/n*100:.0f}%)')

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx]   # (T, C, H, W)
        mask = self.masks[idx]
        img  = img.permute(1, 0, 2, 3)  # (C, T, H, W)

        if self.augment:
            if torch.rand(1) > 0.5:
                img  = torch.flip(img,  [-1])
                mask = torch.flip(mask, [-1])
            if torch.rand(1) > 0.5:
                img  = torch.flip(img,  [-2])
                mask = torch.flip(mask, [-2])
            if torch.rand(1) > 0.5:
                k    = torch.randint(1, 4, (1,)).item()
                img  = torch.rot90(img,  k, [-2,-1])
                mask = torch.rot90(mask, k, [-2,-1])
        return {"image": img, "mask": mask}

print('데이터 로딩...')
train_ds = HLSDataset(PATCH_DIR / 'train.npz', augment=True)
val_ds   = HLSDataset(PATCH_DIR / 'val.npz',   augment=False)
test_ds  = HLSDataset(PATCH_DIR / 'test.npz',  augment=False)

weights = np.where(train_ds.has_fault, 30.0, 1.0)
sampler = WeightedRandomSampler(
    torch.from_numpy(weights).float(),
    num_samples=len(weights), replacement=True
)

BATCH         = 8   # 256×256은 메모리 더 사용 → 16→8
LOADER_KWARGS = dict(num_workers=2, pin_memory=True)
train_loader  = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, **LOADER_KWARGS)
val_loader    = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   **LOADER_KWARGS)
test_loader   = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   **LOADER_KWARGS)

batch       = next(iter(train_loader))
fault_ratio = (batch['mask'].sum(dim=(1,2)) > 0).float().mean()
print(f'배치 fault 비율: {fault_ratio:.2f}')
print(f'이미지 shape: {batch["image"].shape}')

# ── 모델 ──────────────────────────────────────────────────────
print('\n모델 로딩...')
from terratorch.models import EncoderDecoderFactory

factory = EncoderDecoderFactory()
model   = factory.build_model(
    task                  = "segmentation",
    backbone              = "prithvi_eo_v2_300",
    backbone_pretrained   = True,
    backbone_in_channels  = 6,
    backbone_num_frames   = 4,
    decoder               = "UperNetDecoder",
    decoder_channels      = 256,
    decoder_scale_modules = True,
    num_classes           = 2,
).to(device)

for param in model.encoder.parameters():
    param.requires_grad = False
for block in model.encoder.blocks[-4:]:
    for param in block.parameters():
        param.requires_grad = True
for param in model.encoder.norm.parameters():
    param.requires_grad = True

frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Frozen: {frozen:,} | Trainable: {trainable:,}')

# ── Loss ──────────────────────────────────────────────────────
class FaultLoss(nn.Module):
    def __init__(self):
        super().__init__()
        w       = torch.tensor([1.0, 10.0], dtype=torch.float32).to(device)
        self.ce = nn.CrossEntropyLoss(weight=w)

    def forward(self, inputs, targets):
        if hasattr(inputs, 'output'):
            inputs = inputs.output
        targets = targets.to(inputs.device)
        ce    = self.ce(inputs, targets)
        prob  = torch.softmax(inputs, dim=1)[:, 1]
        tgt_f = (targets == 1).float()
        inter = (prob * tgt_f).sum(dim=(1,2))
        union = prob.sum(dim=(1,2)) + tgt_f.sum(dim=(1,2))
        dice  = 1.0 - (2*inter + 1) / (union + 1)
        return ce + dice.mean()

def compute_metrics(logits, targets):
    if hasattr(logits, 'output'):
        logits = logits.output
    preds = logits.argmax(1)
    tp = ((preds==1)&(targets==1)).sum().float()
    fp = ((preds==1)&(targets==0)).sum().float()
    fn = ((preds==0)&(targets==1)).sum().float()
    tn = ((preds==0)&(targets==0)).sum().float()
    return {
        'mIoU':      ((tp/(tp+fp+fn+1e-8)+tn/(tn+fp+fn+1e-8))/2).item(),
        'IoU_fault':  (tp/(tp+fp+fn+1e-8)).item(),
        'F1':         (2*tp/(2*tp+fp+fn+1e-8)).item(),
    }

optimizer = torch.optim.AdamW([
    {'params': [p for n,p in model.named_parameters()
                if p.requires_grad and 'decoder' not in n and 'head' not in n],
     'lr': 1e-5},
    {'params': [p for n,p in model.named_parameters()
                if p.requires_grad and ('decoder' in n or 'head' in n)],
     'lr': 5e-4},
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-7
)
scaler = torch.amp.GradScaler('cuda')

# ── 학습 ──────────────────────────────────────────────────────
NUM_EPOCHS   = 100
PATIENCE     = 30
best_iou     = 0.0
patience_cnt = 0

print('\n' + '='*60)
print('Prithvi-300M + HLS 256×256 — Objective 1A')
print('='*60)
t0 = time.time()

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    t_loss = 0.0
    for batch in train_loader:
        imgs  = batch['image'].float().to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = FaultLoss()(out, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
    t_loss /= len(train_loader)
    scheduler.step()

    model.eval()
    vm = {'mIoU':0.0, 'IoU_fault':0.0, 'F1':0.0}
    with torch.no_grad():
        for batch in val_loader:
            imgs  = batch['image'].float().to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
            m = compute_metrics(out, masks)
            for k in vm: vm[k] += m[k]
    for k in vm: vm[k] /= len(val_loader)

    if epoch % 5 == 0 or epoch <= 5:
        elapsed = (time.time()-t0)/60
        print(f'Ep {epoch:3d} | Loss:{t_loss:.4f} | '
              f'mIoU:{vm["mIoU"]:.4f} | '
              f'IoU_fault:{vm["IoU_fault"]:.4f} | '
              f'F1:{vm["F1"]:.4f} | {elapsed:.1f}min')

    if vm['IoU_fault'] > best_iou:
        best_iou     = vm['IoU_fault']
        patience_cnt = 0
        torch.save({'epoch': epoch,
                    'model_state': model.state_dict(),
                    'val_metrics': vm},
                   CKPT_DIR / 'best_prithvi_256.pth')
        print(f'  💾 Best (ep={epoch}, IoU_fault={best_iou:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nDone {(time.time()-t0)/60:.1f}min | Best IoU_fault: {best_iou:.4f}')

Device: cuda
데이터 로딩...
  train.npz: 672개, fault=71(11%)
  val.npz: 143개, fault=15(10%)
  test.npz: 146개, fault=16(11%)
배치 fault 비율: 0.88
이미지 shape: torch.Size([8, 6, 4, 256, 256])

모델 로딩...


Frozen: 253,499,392 | Trainable: 147,854,850

Prithvi-600M + HLS 256×256 — Objective 1A
Ep   1 | Loss:1.5030 | mIoU:0.4299 | IoU_fault:0.0136 | F1:0.0242 | 0.2min
  💾 Best (ep=1, IoU_fault=0.0136)
Ep   2 | Loss:1.3284 | mIoU:0.4190 | IoU_fault:0.0147 | F1:0.0260 | 0.4min
  💾 Best (ep=2, IoU_fault=0.0147)
Ep   3 | Loss:1.2744 | mIoU:0.3680 | IoU_fault:0.0166 | F1:0.0289 | 0.6min
  💾 Best (ep=3, IoU_fault=0.0166)
Ep   4 | Loss:1.2133 | mIoU:0.4052 | IoU_fault:0.0173 | F1:0.0299 | 0.9min
  💾 Best (ep=4, IoU_fault=0.0173)
Ep   5 | Loss:1.1562 | mIoU:0.3231 | IoU_fault:0.0133 | F1:0.0237 | 1.1min
  💾 Best (ep=6, IoU_fault=0.0196)
Ep  10 | Loss:0.9835 | mIoU:0.4672 | IoU_fault:0.0179 | F1:0.0308 | 2.2min
Ep  15 | Loss:0.8230 | mIoU:0.4937 | IoU_fault:0.0145 | F1:0.0255 | 3.0min
  💾 Best (ep=17, IoU_fault=0.0205)
Ep  20 | Loss:0.6726 | mIoU:0.4671 | IoU_fault:0.0184 | F1:0.0314 | 4.0min
  💾 Best (ep=22, IoU_fault=0.0222)
Ep  25 | Loss:0.5949 | mIoU:0.4948 | IoU_fault:0.0166 | F1:0.0288 | 4.9m

KeyboardInterrupt: 

In [3]:
# ep42 best checkpoint 로드 후 threshold 분석
val_loader = DataLoader(val_ds, batch_size=8,
                        shuffle=False, num_workers=0)

ckpt = torch.load(CKPT_DIR / 'best_prithvi_256.pth',
                  map_location=device)
model.load_state_dict(ckpt['model_state'])

model.eval()
all_probs, all_masks = [], []
with torch.no_grad():
    for batch in val_loader:
        imgs  = batch['image'].float().to(device)
        masks = batch['mask'].to(device)
        with torch.amp.autocast('cuda'):
            out = model(imgs)
        if hasattr(out, 'output'):
            logits = out.output
        probs = logits.softmax(1)[:,1].cpu().numpy()
        all_probs.append(probs)
        all_masks.append(masks.cpu().numpy())

all_probs = np.concatenate(all_probs).ravel()
all_masks = np.concatenate(all_masks).ravel()

print('\nThreshold  IoU_fault   F1')
print('-'*35)
best_t, best_iou = 0.5, 0.0
for t in np.arange(0.05, 0.55, 0.05):
    pred = (all_probs > t).astype(int)
    tp = ((pred==1)&(all_masks==1)).sum()
    fp = ((pred==1)&(all_masks==0)).sum()
    fn = ((pred==0)&(all_masks==1)).sum()
    iou = tp / (tp+fp+fn+1e-8)
    f1  = 2*tp / (2*tp+fp+fn+1e-8)
    print(f'  {t:.2f}      {iou:.4f}     {f1:.4f}')
    if iou > best_iou:
        best_iou, best_t = iou, t

print(f'\n최적 threshold: {best_t:.2f} (IoU_fault={best_iou:.4f})')


Threshold  IoU_fault   F1
-----------------------------------
  0.05      0.0631     0.1187
  0.10      0.0661     0.1241
  0.15      0.0687     0.1285
  0.20      0.0708     0.1322
  0.25      0.0726     0.1354
  0.30      0.0743     0.1384
  0.35      0.0760     0.1413
  0.40      0.0776     0.1441
  0.45      0.0791     0.1465
  0.50      0.0802     0.1485

최적 threshold: 0.50 (IoU_fault=0.0802)
